#### Importing required libraries 

In [15]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score, precision_score,f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
file_path_replies = r"../replies_sydneysiege.pkl"
file_path_posts = r"../posts_sydneysiege.pkl"

#### Testing a single load 

In [7]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=3*24*60)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [8]:
train.head()

,followers,favorite_count,retweet_count,first_time_diff,replies,no_verified,verified,embeddings_avg,rumour,min_since_fst_post
0,-0.129424,-0.252174,1.517832,1.515602,0.375,0,1,"[0.04698175168596208, -0.18934187246486545, -0...",1,5.62
1,-0.128771,-0.617391,-0.171184,3.964339,-0.500,0,1,"[0.1690062526613474, -0.11465575313195586, 0.1...",1,7.73
2,-0.092106,-0.382609,0.131241,-0.023774,0.750,0,1,"[-0.23375208879059012, -0.19412890686230225, -...",1,9.52
3,0.091430,-0.643478,-0.405136,-0.380386,-0.500,0,1,"[0.029154916604359944, -0.25932883098721504, -...",1,9.90
4,-0.129424,0.008696,2.773181,1.384844,0.625,0,1,"[-0.03537977912596294, -0.043058500226054876, ...",1,11.85


In [9]:
previous_node_count = 0

In [10]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 352


In [11]:




# Model definition
class RumorDetectionLSTM(nn.Module):


    """
    Hybrid neural network model for rumor detection using embeddings and structured features.

    This model combines an LSTM network that processes precomputed text embeddings
    with fully connected layers that process additional handcrafted or metadata features.
    The outputs from these two branches are fused and passed through dense layers
    for binary rumor classification.

    Architecture Overview:
    ---------------------
    • Input features are divided into:
        - First 8 features: handcrafted / numerical indicators
        - Last 100 features: embedding vector representing textual content
    • A single-step LSTM processes the embedding to extract contextual semantics
    • Dense layers extract nonlinear structure from auxiliary features
    • Concatenated representation is classified with fully connected layers

    Parameters
    ----------
    embedding_dim : int, default=100
        Dimensionality of the input embedding vector.
    lstm_hidden_size : int, default=32
        Number of hidden units in the LSTM layer.
    dense_hidden_size : int, default=16
        Number of hidden units in the dense feature branch.

    Forward Input
    -------------
    x : torch.Tensor
        Tensor of shape (batch_size, 108) where:
        - x[:, :8] are handcrafted features
        - x[:, -100:] is a sentence/document embedding

    Returns
    -------
    torch.Tensor
        A 1D tensor of shape (batch_size,) containing probabilities in [0, 1]
        where values close to 1 indicate high likelihood of rumor.

    Notes
    -----
    - Assumes sequence length = 1 in embedding branch.
    - Uses sigmoid activation for binary classification tasks.
    """
    
    def __init__(self, embedding_dim=100, lstm_hidden_size=32, dense_hidden_size=16):
        super(RumorDetectionLSTM, self).__init__()
        
        # LSTM for the 100-dimensional embeddings
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=lstm_hidden_size, batch_first=True)
        
        # Dense layers for other features
        self.dense1 = nn.Linear(8, 16)  # 8 non-embedding features
        self.dense2 = nn.Linear(16, dense_hidden_size)
        
        # Combine LSTM and dense features
        self.fc1 = nn.Linear(lstm_hidden_size + dense_hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # Separate embeddings and other features
        embeddings = x[:, -100:].unsqueeze(1)  # (batch, seq_len=1, embedding_dim)
        other_features = x[:, :8]  # First 8 features
        
        # LSTM output
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = lstm_out[:, -1, :]  # Get the last LSTM output
        
        # Dense layers for other features
        dense_out = torch.relu(self.dense1(other_features))
        dense_out = torch.relu(self.dense2(dense_out))
        
        # Concatenate LSTM and dense outputs
        combined = torch.cat((lstm_out, dense_out), dim=1)
        
        # Fully connected layers for classification
        x = torch.relu(self.fc1(combined))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze()


#### Example  training

In [12]:
# Assuming X_train, X_test, y_train, and y_test are available as numpy arrays
# Convert them to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [7]:


# Model, criterion, optimizer initialization (as before)
model = RumorDetectionLSTM()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with loss and recall monitoring
epochs = 75  # Adjust as needed
train_recall_interval = 50  # Calculate train recall every 10 epochs
loss_interval = 50  # Print loss every 10 epochs

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Print loss every 10 epochs
    if (epoch + 1) % loss_interval == 0:
        model.eval()
        train_preds = []
        train_labels = []
        with torch.no_grad():
            for X_batch, y_batch in train_loader:
                output = model(X_batch)
                preds = (output >= 0.5).int()  # Binarize predictions
                train_preds.extend(preds.tolist())
                train_labels.extend(y_batch.tolist())
        
        train_recall = recall_score(train_labels, train_preds)
        train_precision = precision_score(train_labels, train_preds)
        
print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f},\
              Train Precision: {train_precision:.4f},Train Recall: {train_recall:.4f}")
    


# Final evaluation on test set with recall and precision
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = (output >= 0.5).int()  # Binarize predictions
        test_preds.extend(preds.tolist())
        test_labels.extend(y_batch.tolist())

# Calculate final test recall and precision
test_recall = recall_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)

print(f"Final Test Recall: {test_recall:.4f}")
print(f"Final Test Precision: {test_precision:.4f}")


Epoch 75, Train Loss: 0.0153,              Train Precision: 0.9760,Train Recall: 0.9486
Final Test Recall: 1.0000
Final Test Precision: 0.2000


In [19]:
# Model, criterion, optimizer initialization (as before)
model = RumorDetectionLSTM()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 200
loss_interval = 50

# Save train outputs for threshold tuning
train_probs_all = []
train_labels_all = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # Store train outputs for final threshold selection
    if (epoch + 1) == epochs:
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in train_loader:
                output = model(X_batch)
                train_probs_all.extend(output.squeeze().tolist())
                train_labels_all.extend(y_batch.squeeze().tolist())
    
    # Logging
    if (epoch + 1) % loss_interval == 0:
        print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f}")

# ------------------------
# Find best threshold on train set
# ------------------------
train_probs_all = np.array(train_probs_all)
train_labels_all = np.array(train_labels_all)

best_thresh = 0.5
best_f1 = 0.0
for t in np.arange(0.0, 1.01, 0.01):
    preds = (train_probs_all >= t).astype(int)
    f1 = f1_score(train_labels_all, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\nBest Threshold on Train (max F1): {best_thresh:.2f} | F1: {best_f1:.4f}")

# ------------------------
# Final evaluation on test set using best threshold
# ------------------------
model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        test_probs.extend(output.squeeze().tolist())
        test_labels.extend(y_batch.squeeze().tolist())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds = (test_probs >= best_thresh).astype(int)

# Calculate final metrics
test_recall = recall_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)

print(f"Final Test Recall: {test_recall:.4f}")
print(f"Final Test Precision: {test_precision:.4f}")
print(f"Final Test F1: {test_f1:.4f}")


Epoch 50, Train Loss: 0.1344
Epoch 100, Train Loss: 0.0401
Epoch 150, Train Loss: 0.0059
Epoch 200, Train Loss: 0.0040

Best Threshold on Train (max F1): 0.15 | F1: 1.0000
Final Test Recall: 0.8000
Final Test Precision: 0.6486
Final Test F1: 0.7164


#### Setting MLflow Experiment

In [16]:
mlflow.set_experiment("LSTM 2025-11-04 Sydney Siege")

2025/11/09 17:39:19 INFO mlflow.tracking.fluent: Experiment with name 'LSTM 2025-11-04 Sydney Siege' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/92', creation_time=1762709959356, experiment_id='92', last_update_time=1762709959356, lifecycle_stage='active', name='LSTM 2025-11-04 Sydney Siege', tags={}>

#### Loading dataset statistics to get the final time cut 

In [17]:
df_posts_by_tm = pd.read_csv('sydneysiege_posts_by_time_cut.csv')

df_posts_by_tm['new_posts_cum_sum'] = df_posts_by_tm.new_posts.cumsum()

max_time_cut = int(df_posts_by_tm[df_posts_by_tm.new_posts_cum_sum==int(df_posts_by_tm.new_posts_cum_sum.max())]\
                   .time_cut.min())

In [ ]:
previous_node_count = 0

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\nProcessing time_cut: {time_cut}")

    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train,test= processor.get_final_dataframes()


    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    print(f"New Instances: {X_test_new.shape[0]}")


    # Handle class imbalance
    num_pos = sum(y_train)
    num_neg = len(y_train) - num_pos
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    X_test_new = torch.tensor(X_test_new, dtype=torch.float32)
    y_test = torch.tensor(y_test.values, dtype=torch.float32)
    y_test_new = torch.tensor(y_test_new.values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    test_dataset_new= TensorDataset(X_test_new, y_test_new)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    test_loader_new = DataLoader(test_dataset_new, batch_size=32)

    # Define model, loss, and optimizer
    model = RumorDetectionLSTM()  # <- define your model class elsewhere
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    epochs = 200
    loss_interval = 50

    train_probs_all = []
    train_labels_all = []

    with mlflow.start_run():
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model(X_batch).view(-1)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            if (epoch + 1) % loss_interval == 0:
                print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f}")

            # Store outputs for threshold tuning after last epoch
            if (epoch + 1) == epochs:
                model.eval()
                with torch.no_grad():
                    for X_batch, y_batch in train_loader:
                        output = model(X_batch).view(-1)
                        train_probs_all.extend(output.tolist())
                        train_labels_all.extend(y_batch.tolist())

        # Threshold tuning (maximize F1)
        train_probs_all = np.array(train_probs_all)
        train_labels_all = np.array(train_labels_all)
        best_thresh = 0.5
        best_f1 = 0.0
        for t in np.arange(0.0, 1.01, 0.01):
            preds = (train_probs_all >= t).astype(int)
            f1 = f1_score(train_labels_all, preds)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t

        print(f"\nBest Threshold on Train (max F1): {best_thresh:.2f} | F1: {best_f1:.4f}")

        # Final test evaluation
        model.eval()
        test_probs = []
        test_labels = []

        test_probs_new = []
        test_labels_new = []
        
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch).view(-1)
                test_probs.extend(output.tolist())
                test_labels.extend(y_batch.tolist())
                
            if X_test_new.shape[0] >0:
                for X_batch, y_batch in test_loader_new:
                    output = model(X_batch).view(-1)
                    test_probs_new.extend(output.tolist())
                    test_labels_new.extend(y_batch.tolist())

                test_probs_new = np.array(test_probs_new)
                test_labels_new = np.array(test_labels_new)
                test_preds_new  = (test_probs_new  >= best_thresh).astype(int)
                test_recall_new = recall_score(test_labels_new, test_preds_new)
                test_precision_new = precision_score(test_labels_new, test_preds_new)
                test_f1_new = f1_score(test_labels_new, test_preds_new)
                test_acc_new = accuracy_score(test_labels_new, test_preds_new)
                mlflow.log_metric("curr_precision", test_precision_new)
                mlflow.log_metric("curr_recall", test_recall_new)
                mlflow.log_metric("curr_acc", test_acc_new)
                mlflow.log_metric("curr_f1", test_f1_new)
            else:
                mlflow.log_metric("curr_precision", 0)
                mlflow.log_metric("curr_recall", 0)
                mlflow.log_metric("curr_acc", 0)
                mlflow.log_metric("curr_f1", 0)

        test_probs = np.array(test_probs)
        test_labels = np.array(test_labels)
        test_preds = (test_probs >= best_thresh).astype(int)
     

        test_recall = recall_score(test_labels, test_preds)
        test_precision = precision_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds)
        test_auc = roc_auc_score(test_labels, test_probs)
        test_acc = accuracy_score(test_labels, test_preds)

        print(f"Final Test Recall: {test_recall:.4f}")
        print(f"Final Test Precision: {test_precision:.4f}")
        print(f"Final Test F1: {test_f1:.4f}")
        print(f"Final Test AUC: {test_auc:.4f}")

        # MLflow logging
        mlflow.log_metric("train_f1", best_f1)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_auc", test_auc)
        mlflow.log_metric("time_cut", time_cut)
        mlflow.log_param("learning_rate", 0.001)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("threshold", best_thresh)



Processing time_cut: 40
New Instances: 30
Epoch 50, Train Loss: 0.2605
Epoch 100, Train Loss: 0.1784
Epoch 150, Train Loss: 0.0809
Epoch 200, Train Loss: 0.0243

Best Threshold on Train (max F1): 0.31 | F1: 0.9913
Final Test Recall: 0.1364
Final Test Precision: 1.0000
Final Test F1: 0.2400
Final Test AUC: 0.7159

Processing time_cut: 70
New Instances: 18
Epoch 50, Train Loss: 0.2908
Epoch 100, Train Loss: 0.2412
Epoch 150, Train Loss: 0.1277
Epoch 200, Train Loss: 0.0534

Best Threshold on Train (max F1): 0.35 | F1: 0.9737


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Final Test Recall: 0.0882
Final Test Precision: 1.0000
Final Test F1: 0.1622
Final Test AUC: 0.6324

Processing time_cut: 100
New Instances: 20
Epoch 50, Train Loss: 0.2372
Epoch 100, Train Loss: 0.2097
Epoch 150, Train Loss: 0.1182
Epoch 200, Train Loss: 0.0844

Best Threshold on Train (max F1): 0.39 | F1: 0.9821


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Final Test Recall: 0.0500
Final Test Precision: 1.0000
Final Test F1: 0.0952
Final Test AUC: 0.7223

Processing time_cut: 130
New Instances: 20
Epoch 50, Train Loss: 0.2304
Epoch 100, Train Loss: 0.1684
Epoch 150, Train Loss: 0.1016
Epoch 200, Train Loss: 0.0235

Best Threshold on Train (max F1): 0.25 | F1: 0.9956
Final Test Recall: 0.1739
Final Test Precision: 0.8889
Final Test F1: 0.2909
Final Test AUC: 0.7376

Processing time_cut: 160
New Instances: 14
Epoch 50, Train Loss: 0.2428
Epoch 100, Train Loss: 0.1610
Epoch 150, Train Loss: 0.0822
Epoch 200, Train Loss: 0.0255

Best Threshold on Train (max F1): 0.31 | F1: 0.9913
Final Test Recall: 0.1346
Final Test Precision: 0.7000
Final Test F1: 0.2258
Final Test AUC: 0.6523

Processing time_cut: 190
New Instances: 12
Epoch 50, Train Loss: 0.2601
Epoch 100, Train Loss: 0.2072
Epoch 150, Train Loss: 0.1300
Epoch 200, Train Loss: 0.0688

Best Threshold on Train (max F1): 0.57 | F1: 0.9596
Final Test Recall: 0.0702
Final Test Precision: 0.80

...

Final Test Recall: 0.1329
Final Test Precision: 0.6786
Final Test F1: 0.2222
Final Test AUC: 0.7086

Processing time_cut: 610
New Instances: 1
Epoch 50, Train Loss: 0.2553
Epoch 100, Train Loss: 0.1945
Epoch 150, Train Loss: 0.1156
Epoch 200, Train Loss: 0.1043

Best Threshold on Train (max F1): 0.36 | F1: 0.9780


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Final Test Recall: 0.2238
Final Test Precision: 0.8000
Final Test F1: 0.3497
Final Test AUC: 0.6876

Processing time_cut: 640
New Instances: 0
Epoch 50, Train Loss: 0.3051
Epoch 100, Train Loss: 0.2031
Epoch 150, Train Loss: 0.1201
Epoch 200, Train Loss: 0.0714

Best Threshold on Train (max F1): 0.25 | F1: 0.9700
Final Test Recall: 0.2587
Final Test Precision: 0.7708
Final Test F1: 0.3874
Final Test AUC: 0.6967

Processing time_cut: 670
New Instances: 0
Epoch 50, Train Loss: 0.2924
Epoch 100, Train Loss: 0.1913
Epoch 150, Train Loss: 0.1314
Epoch 200, Train Loss: 0.0590

Best Threshold on Train (max F1): 0.40 | F1: 0.9780
Final Test Recall: 0.1259
Final Test Precision: 0.7200
Final Test F1: 0.2143
Final Test AUC: 0.6376

Processing time_cut: 700
New Instances: 1
Epoch 50, Train Loss: 0.3661
Epoch 100, Train Loss: 0.2099
Epoch 150, Train Loss: 0.1198
Epoch 200, Train Loss: 0.0526

Best Threshold on Train (max F1): 0.33 | F1: 0.9825


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Final Test Recall: 0.2098
Final Test Precision: 0.8108
Final Test F1: 0.3333
Final Test AUC: 0.6822

Processing time_cut: 730
New Instances: 0
Epoch 50, Train Loss: 0.2678
Epoch 100, Train Loss: 0.1842
Epoch 150, Train Loss: 0.1072
Epoch 200, Train Loss: 0.0462

Best Threshold on Train (max F1): 0.52 | F1: 0.9868
Final Test Recall: 0.1678
Final Test Precision: 0.8000
Final Test F1: 0.2775
Final Test AUC: 0.6774

Processing time_cut: 760
New Instances: 0
Epoch 50, Train Loss: 0.2762
Epoch 100, Train Loss: 0.1858
Epoch 150, Train Loss: 0.1016
Epoch 200, Train Loss: 0.0282

Best Threshold on Train (max F1): 0.27 | F1: 0.9956
Final Test Recall: 0.1958
Final Test Precision: 0.7179
Final Test F1: 0.3077
Final Test AUC: 0.6726

Processing time_cut: 790
New Instances: 0
Epoch 50, Train Loss: 0.3019
Epoch 100, Train Loss: 0.2018
Epoch 150, Train Loss: 0.1741
Epoch 200, Train Loss: 0.0957

Best Threshold on Train (max F1): 0.29 | F1: 0.9511
Final Test Recall: 0.3357
Final Test Precision: 0.8571
